# Swing lows — the mirror test

**Pre-declared, before the first run on real data:**

| | prediction | basis |
|---|---|---|
| hold rate | 27–33% | highs gave 30.0%, band is ±3 points |
| confirmation lag | 1.4–1.7% | highs gave 1.57%; structural, so this checks the code |
| adverse excursion | no prediction | no basis for one |

Structure only. No predictor testing — re-mining eleven predictors in the
other direction on the same data is how a false positive gets found.

## Write `src/pivot_lows.py`

In [1]:
import pathlib

content = r'''"""
Swing lows — the mirror of pivot_auto.find_pivots.

The study tested swing highs only. This asks whether lows behave the same way,
and nothing else: structure only, no predictor testing. Re-mining eleven
predictors on the same data in the other direction is how a false positive gets
found, and the point of this file is a single pre-declared comparison.

PRE-DECLARED, before the first run:
    hold rate         27-33%   (highs gave 30.0%, band is +/- 3 points)
    confirmation lag  1.4-1.7% (highs gave 1.57%; structural, so this is a
                                check on the code rather than on the market)
    adverse excursion no prediction -- no basis for one

Every rule mirrors pivot_auto exactly:
    fractal        low[i] < low[i-1] and low[i] < low[i+1]
    leg            running HIGH since the last qualified low
    qualification  (leg_high - low[i]) >= atr_mult * ATR[i], min_sep bars apart
    span           until a CLOSE below the pivot low, or span_bars
    wicks          count for max_down / max_rise, as in the highs

COLUMN MAPPING against pivot_auto
    max_up   -> max_down   adverse excursion, now BELOW the pivot low
    max_drop -> max_rise   the favourable move, now upward
    gain     -> decline    the leg into the pivot, now a fall
Both adverse columns are positive numbers measuring distance the wrong way.
"""

import numpy as np
import pandas as pd

from pivot_auto import atr

__all__ = ["find_lows", "summarise_lows", "compare"]


def find_lows(df, atr_mult=3.0, atr_len=14, min_sep=2, span_bars=32,
              rise_target=3.5):
    """One row per qualified swing low, with its outcome fields."""
    df = df.reset_index(drop=True).copy()
    df["atr"] = atr(df, atr_len)
    high, low, close = df["high"].values, df["low"].values, df["close"].values
    atrv = df["atr"].values
    n = len(df)

    leg_hi_price, leg_hi_bar, leg_bars, q_lo_bar = np.nan, -1, 0, -1
    out = []
    for i in range(1, n - 1):
        is_fractal = low[i] < low[i - 1] and low[i] < low[i + 1]
        if is_fractal and not np.isnan(atrv[i]):
            sep_ok = q_lo_bar < 0 or (i - q_lo_bar) >= min_sep
            disp_ok = np.isnan(leg_hi_price) or (leg_hi_price - low[i]) >= atr_mult * atrv[i]
            order_ok = leg_hi_bar < 0 or leg_hi_bar < i
            if sep_ok and disp_ok and order_ok and not np.isnan(leg_hi_price):
                ref = low[i]
                decline = (leg_hi_price - ref) / leg_hi_price * 100
                mult = (leg_hi_price - ref) / atrv[i]
                hi_seen, lo_seen = ref, ref
                bars_held, bars_to_35 = np.nan, np.nan
                for k in range(1, span_bars + 1):
                    j = i + k
                    if j >= n:
                        break
                    hi_seen = max(hi_seen, high[j])
                    lo_seen = min(lo_seen, low[j])
                    if np.isnan(bars_to_35) and high[j] >= ref * (1 + rise_target / 100):
                        bars_to_35 = k
                    if close[j] < ref:
                        bars_held = k
                        break
                out.append({
                    "date": df["date"].iloc[i], "bar": i,
                    "atr_mult": round(mult, 2), "decline": round(decline, 2),
                    "bars": leg_bars,
                    "max_down": round(max(ref - lo_seen, 0) / ref * 100, 2),
                    "max_rise": round(max(hi_seen - ref, 0) / ref * 100, 2),
                    "lag_pct": round((close[i + 1] - ref) / ref * 100, 2),
                    "bars_held": bars_held, "bars_to_35": bars_to_35})
                leg_hi_price, leg_hi_bar, leg_bars = np.nan, -1, 0
                q_lo_bar = i
        if np.isnan(leg_hi_price) or high[i] > leg_hi_price:
            leg_hi_price, leg_hi_bar, leg_bars = high[i], i, 0
        else:
            leg_bars += 1
    return pd.DataFrame(out)


def summarise_lows(p, rise_target=3.5):
    """Same shape as pivot_auto.summarise, with the directions flipped."""
    from scipy import stats
    p = p.copy()
    p["held"] = p["bars_held"].isna()
    p["ratio"] = p["max_rise"] / p["decline"]
    p["span"] = p["bars_held"].fillna(32)
    print(f"\n{len(p)} swing lows   {p['date'].min().date()} to {p['date'].max().date()}")
    print("=" * 68)
    print(f"  held the full span: {p['held'].sum()}/{len(p)} ({p['held'].mean()*100:.0f}%)")
    print(f"  reached +{rise_target}%:      {p['bars_to_35'].notna().sum()}/{len(p)} "
          f"({p['bars_to_35'].notna().mean()*100:.0f}%)")
    print("\n  THE TWO POPULATIONS")
    print("  " + "-" * 64)
    print(f"  {'':<12}{'n':>5}{'med rise':>11}{'med ratio':>11}{'med down':>11}{'med decline':>13}")
    for lab, g in [("held", p[p["held"]]), ("failed", p[~p["held"]])]:
        if len(g):
            print(f"  {lab:<12}{len(g):>5}{g['max_rise'].median():>11.2f}"
                  f"{g['ratio'].median():>11.2f}{g['max_down'].median():>11.2f}"
                  f"{g['decline'].median():>13.2f}")
    if "regime" in p.columns:
        print("\n  HOLD RATE BY REGIME")
        print("  " + "-" * 64)
        for r, g in p.groupby("regime"):
            print(f"  {r:<16}{len(g):>5} lows   held {g['held'].mean()*100:>3.0f}%   "
                  f"med rise {g['max_rise'].median():>5.2f}%")
    print("\n  ADVERSE EXCURSION (below the pivot low)")
    print("  " + "-" * 64)
    u = p["max_down"]
    print("  " + "   ".join(f"p{q}: {np.percentile(u, q):.2f}%" for q in (50, 80, 90, 95)))
    print(f"  never traded below the pivot low: {(u == 0).sum()}/{len(p)}")
    print("\n  CONFIRMATION LAG")
    print("  " + "-" * 64)
    print(f"  mean {p['lag_pct'].mean():.2f}%  -- gone before the low is knowable")
    print("\n  SPAN CONFOUND CHECK")
    print("  " + "-" * 64)
    rho, _ = stats.spearmanr(p["span"], p["max_rise"])
    print(f"  span vs max_rise: rho={rho:+.3f}")
    return p


def compare(highs, lows, df):
    """The pre-declared comparison. Prints pass/fail against the two bands.

    `highs` comes from pivot_auto.find_pivots and has no lag column, so the
    confirmation lag is recomputed here from the price data the same way.
    """
    df = df.reset_index(drop=True)
    hi, cl = df["high"].values, df["close"].values
    h_held = highs["bars_held"].isna()
    l_held = lows["bars_held"].isna()
    h_lag = np.mean([(hi[int(b)] - cl[int(b) + 1]) / hi[int(b)] * 100
                     for b in highs["bar"] if int(b) + 1 < len(df)])

    rows = [
        ("n", len(highs), len(lows), ""),
        ("hold rate %", h_held.mean() * 100, l_held.mean() * 100, "27-33"),
        ("adverse p50 %", highs["max_up"].median(), lows["max_down"].median(), "none"),
        ("adverse p80 %", highs["max_up"].quantile(.8), lows["max_down"].quantile(.8), "none"),
        ("med move, held", highs[h_held]["max_drop"].median(),
         lows[l_held]["max_rise"].median(), ""),
        ("med move, failed", highs[~h_held]["max_drop"].median(),
         lows[~l_held]["max_rise"].median(), ""),
        ("confirm lag %", h_lag, lows["lag_pct"].mean(), "1.4-1.7"),
    ]
    print(f"{'':<20}{'highs':>10}{'lows':>10}{'predicted':>12}{'':>8}")
    print("-" * 60)
    for name, a, b, band in rows:
        verdict = ""
        if band == "27-33":
            verdict = "ok" if 27 <= b <= 33 else "MISS"
        elif band == "1.4-1.7":
            verdict = "ok" if 1.4 <= b <= 1.7 else "MISS"
        av = f"{a:>10.2f}" if isinstance(a, float) and np.isfinite(a) else f"{a:>10}"
        print(f"{name:<20}{av}{b:>10.2f}{band:>12}{verdict:>8}")
'''

pathlib.Path('src/pivot_lows.py').write_text(content, encoding='utf-8')
print('wrote src/pivot_lows.py')

wrote src/pivot_lows.py


## Run it

In [2]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
from pivot_auto import find_pivots, add_regime
from pivot_lows import find_lows, summarise_lows, compare

df = pd.read_csv('data/btc_6h.csv', parse_dates=['date'])

highs = find_pivots(df, atr_mult=1.5)
lows  = find_lows(df,   atr_mult=1.5)
lows  = add_regime(lows, df)
lows  = summarise_lows(lows)


622 swing lows   2023-08-01 to 2026-08-28
  held the full span: 233/622 (37%)
  reached +3.5%:      384/622 (62%)

  THE TWO POPULATIONS
  ----------------------------------------------------------------
                  n   med rise  med ratio   med down  med decline
  held          233      10.04       2.37       0.00         3.91
  failed        389       2.99       0.78       1.24         3.54

  HOLD RATE BY REGIME
  ----------------------------------------------------------------
  consolidation     336 lows   held  38%   med rise  4.59%
  down              125 lows   held  36%   med rise  5.20%
  unknown            10 lows   held  50%   med rise  3.30%
  up                151 lows   held  38%   med rise  4.30%

  ADVERSE EXCURSION (below the pivot low)
  ----------------------------------------------------------------
  p50: 0.84%   p80: 1.75%   p90: 2.54%   p95: 3.35%
  never traded below the pivot low: 163/622

  CONFIRMATION LAG
  -------------------------------------------

## The pre-declared comparison

In [3]:
compare(highs, lows, df)

                         highs      lows   predicted        
------------------------------------------------------------
n                          615    622.00                    
hold rate %              29.92     37.46       27-33    MISS
adverse p50 %             0.80      0.84        none        
adverse p80 %             1.78      1.75        none        
med move, held            9.91     10.04                    
med move, failed          2.66      2.99                    
confirm lag %             1.57      1.90     1.4-1.7    MISS


## Record the verdict

Write what happened here, before interpreting it. If hold rate landed
outside 27–33%, that is an asymmetry and worth its own section in
FINDINGS.md. If confirmation lag landed outside 1.4–1.7%, suspect the code
before the market — that quantity is structural.

*(verdict: )*

In [4]:
import numpy as np, pandas as pd

hi, lo, cl = df["high"].values, df["low"].values, df["close"].values

# per-pivot confirmation lag for the highs, computed the same way as the lows
h_lag = pd.Series([(hi[int(b)] - cl[int(b)+1]) / hi[int(b)] * 100
                   for b in highs["bar"] if int(b)+1 < len(df)])
l_lag = lows["lag_pct"]

print("CONFIRMATION LAG -- mean vs median")
print("-" * 52)
print(f"{'':<10}{'mean':>9}{'median':>9}{'p80':>9}{'n':>8}")
for name, s in [("highs", h_lag), ("lows", l_lag)]:
    print(f"{name:<10}{s.mean():>9.2f}{s.median():>9.2f}{s.quantile(.8):>9.2f}{len(s):>8}")

tot = (cl[-1] - cl[0]) / cl[0] * 100
print(f"\nSAMPLE DRIFT")
print("-" * 52)
print(f"  total return over the window: {tot:+.0f}%")

highs = highs.copy(); lows = lows.copy()
if "regime" not in highs.columns:
    highs = add_regime(highs, df)
highs["held"] = highs["bars_held"].isna()
lows["held"]  = lows["bars_held"].isna()
highs["year"] = pd.to_datetime(highs["date"]).dt.year
lows["year"]  = pd.to_datetime(lows["date"]).dt.year

yr_ret = (df.assign(year=df["date"].dt.year)
            .groupby("year")["close"].agg(lambda s: (s.iloc[-1]-s.iloc[0])/s.iloc[0]*100))

print("\nHOLD RATE BY YEAR -- does the gap track that year's return?")
print("-" * 52)
print(f"{'year':<8}{'return':>9}{'highs':>9}{'lows':>9}{'gap':>8}{'n_hi':>7}{'n_lo':>7}")
for y in sorted(set(highs["year"]) & set(lows["year"])):
    a = highs.loc[highs["year"] == y, "held"]
    b = lows.loc[lows["year"] == y, "held"]
    print(f"{y:<8}{yr_ret.get(y, float('nan')):>+9.0f}{a.mean()*100:>9.1f}"
          f"{b.mean()*100:>9.1f}{(b.mean()-a.mean())*100:>+8.1f}{len(a):>7}{len(b):>7}")

print("\nHOLD RATE BY REGIME -- weaker evidence, regime is a label not a return")
print("-" * 52)
print(f"{'regime':<16}{'highs':>9}{'lows':>9}{'gap':>8}{'n_hi':>7}{'n_lo':>7}")
for r in ["up", "consolidation", "down"]:
    a = highs.loc[highs["regime"] == r, "held"]
    b = lows.loc[lows["regime"] == r, "held"]
    if len(a) and len(b):
        print(f"{r:<16}{a.mean()*100:>9.1f}{b.mean()*100:>9.1f}"
              f"{(b.mean()-a.mean())*100:>+8.1f}{len(a):>7}{len(b):>7}")

CONFIRMATION LAG -- mean vs median
----------------------------------------------------
               mean   median      p80       n
highs          1.57     1.27     2.25     615
lows           1.90     1.46     2.74     622

SAMPLE DRIFT
----------------------------------------------------
  total return over the window: +165%

HOLD RATE BY YEAR -- does the gap track that year's return?
----------------------------------------------------
year       return    highs     lows     gap   n_hi   n_lo
2023          +44     23.1     47.3   +24.2     78     91
2024         +121     29.6     38.1    +8.5    206    202
2025           -6     33.0     34.7    +1.7    197    193
2026          -11     29.9     33.8    +4.0    134    136

HOLD RATE BY REGIME -- weaker evidence, regime is a label not a return
----------------------------------------------------
regime              highs     lows     gap   n_hi   n_lo
up                   29.5     37.7    +8.2    193    151
consolidation        30.4 

In [5]:
highs = add_regime(highs, df)
highs["held"] = highs["bars_held"].isna()

print(f"{'regime':<16}{'highs':>9}{'lows':>9}{'gap':>8}{'n_hi':>7}{'n_lo':>7}")
for r in ["up", "consolidation", "down"]:
    a = highs[highs["regime"] == r]["held"]
    b = lows[lows["regime"] == r]["held"]
    if len(a) and len(b):
        print(f"{r:<16}{a.mean()*100:>9.1f}{b.mean()*100:>9.1f}"
              f"{(b.mean()-a.mean())*100:>+8.1f}{len(a):>7}{len(b):>7}")

regime              highs     lows     gap   n_hi   n_lo
up                   29.5     37.7    +8.2    193    151
consolidation        30.4     37.5    +7.1    336    336
down                 25.9     36.0   +10.1     81    125


In [6]:
import numpy as np, pandas as pd
from scipy import stats
from pivot_auto import find_pivots, add_regime
from pivot_lows import find_lows

FLAT_YEARS = [2025, 2026]          # BTC -6% and -11%; the driftless subsample

def _prep(p, df, kind):
    p = add_regime(p.copy(), df)
    p["held"] = p["bars_held"].isna()
    p["year"] = pd.to_datetime(p["date"]).dt.year
    p["adverse"] = p["max_up"] if kind == "high" else p["max_down"]
    p["move"]    = p["max_drop"] if kind == "high" else p["max_rise"]
    return p

def flat(p):
    return p[p["year"].isin(FLAT_YEARS)]

def prop_test(a, b):
    t = [[a.sum(), len(a)-a.sum()], [b.sum(), len(b)-b.sum()]]
    if min(min(r) for r in t) < 1:
        return float("nan")
    return stats.chi2_contingency(t)[1]

print("ATR THRESHOLD SWEEP -- flat years only (2025-2026)")
print("=" * 74)
print(f"{'mult':>6}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}"
      f"{'gap':>8}{'p':>8}{'hi adv':>9}{'lo adv':>9}")
print("-" * 74)
for m in (1.5, 2.0, 2.5, 3.0, 4.0):
    h = flat(_prep(find_pivots(df, atr_mult=m), df, "high"))
    l = flat(_prep(find_lows(df,   atr_mult=m), df, "low"))
    if not len(h) or not len(l):
        continue
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{m:>6.1f}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}"
          f"{h['adverse'].median():>9.2f}{l['adverse'].median():>9.2f}")

print("\nPIVOT SEPARATION SWEEP -- flat years only, atr_mult=1.5")
print("=" * 74)
print(f"{'min_sep':>8}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}"
      f"{'gap':>8}{'p':>8}{'hi adv':>9}{'lo adv':>9}")
print("-" * 74)
for s in (2, 3, 4, 6, 8):
    h = flat(_prep(find_pivots(df, atr_mult=1.5, min_sep=s), df, "high"))
    l = flat(_prep(find_lows(df,   atr_mult=1.5, min_sep=s), df, "low"))
    if not len(h) or not len(l):
        continue
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{s:>8}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}"
          f"{h['adverse'].median():>9.2f}{l['adverse'].median():>9.2f}")

h15 = _prep(find_pivots(df, atr_mult=1.5), df, "high")
l15 = _prep(find_lows(df,   atr_mult=1.5), df, "low")

print("\nWITHIN EACH FLAT YEAR SEPARATELY -- atr_mult=1.5")
print("=" * 74)
print(f"{'year':>6}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
for y in FLAT_YEARS:
    h = h15[h15["year"] == y]; l = l15[l15["year"] == y]
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{y:>6}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}")

print("\nBY REGIME -- flat years only")
print("=" * 74)
print(f"{'regime':<16}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
hf, lf = flat(h15), flat(l15)
for r in ["up", "consolidation", "down"]:
    h = hf[hf["regime"] == r]; l = lf[lf["regime"] == r]
    if len(h) < 5 or len(l) < 5:
        continue
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{r:<16}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}")

print("\nTHE TWO LOOSE ENDS -- full sample vs flat years")
print("=" * 74)
for label, hh, ll in [("full sample", h15, l15), ("flat years", hf, lf)]:
    hn = (hh["adverse"] == 0).mean() * 100
    ln = (ll["adverse"] == 0).mean() * 100
    ht = hh["bars_to_35"].notna().mean() * 100
    lt = ll["bars_to_35"].notna().mean() * 100
    print(f"  {label:<14} never adverse: highs {hn:>4.0f}%  lows {ln:>4.0f}%   "
          f"reached 3.5%: highs {ht:>4.0f}%  lows {lt:>4.0f}%")

print("\nADVERSE EXCURSION -- the symmetric one, does it stay symmetric?")
print("=" * 74)
print(f"{'':<14}{'hi p50':>9}{'lo p50':>9}{'hi p80':>9}{'lo p80':>9}{'n_hi':>7}{'n_lo':>7}")
for label, hh, ll in [("full sample", h15, l15), ("flat years", hf, lf)]:
    print(f"  {label:<12}{hh['adverse'].median():>9.2f}{ll['adverse'].median():>9.2f}"
          f"{hh['adverse'].quantile(.8):>9.2f}{ll['adverse'].quantile(.8):>9.2f}"
          f"{len(hh):>7}{len(ll):>7}")

ATR THRESHOLD SWEEP -- flat years only (2025-2026)
  mult   n_hi   n_lo   hi hold   lo hold     gap       p   hi adv   lo adv
--------------------------------------------------------------------------
   1.5    331    329     31.7%     34.3%    +2.6   0.526     0.70     0.80
   2.0    228    212     31.1%     35.8%    +4.7   0.345     0.70     0.89
   2.5    165    154     32.7%     35.7%    +3.0   0.657     0.68     0.86
   3.0    125    115     33.6%     38.3%    +4.7   0.537     0.70     0.86
   4.0     69     66     36.2%     34.8%    -1.4   1.000     0.66     0.82

PIVOT SEPARATION SWEEP -- flat years only, atr_mult=1.5
 min_sep   n_hi   n_lo   hi hold   lo hold     gap       p   hi adv   lo adv
--------------------------------------------------------------------------
       2    331    329     31.7%     34.3%    +2.6   0.526     0.70     0.80
       3    309    304     31.4%     33.9%    +2.5   0.568     0.73     0.81
       4    287    282     32.1%     33.7%    +1.6   0.745   

In [7]:
import numpy as np, pandas as pd

for name, p, col in [("HIGHS", h15, "max_drop"), ("LOWS", l15, "max_rise")]:
    b = p.loc[~p["held"], "bars_held"]
    t = p.loc[p["bars_to_35"].notna(), "bars_to_35"]
    print(f"\n{name}")
    print("-" * 56)
    print(f"  failed pivots, bars until invalidated   n={len(b)}")
    print("   " + "  ".join(f"p{q}: {np.percentile(b,q):.0f}" for q in (10,25,50,75,90)))
    print(f"   within 2 bars: {(b<=2).mean()*100:.0f}%   "
          f"within 4: {(b<=4).mean()*100:.0f}%   within 8: {(b<=8).mean()*100:.0f}%")
    print(f"  bars until 3.5% favourable move        n={len(t)}")
    print("   " + "  ".join(f"p{q}: {np.percentile(t,q):.0f}" for q in (10,25,50,75,90)))


HIGHS
--------------------------------------------------------
  failed pivots, bars until invalidated   n=431
   p10: 2  p25: 3  p50: 6  p75: 12  p90: 22
   within 2 bars: 15%   within 4: 35%   within 8: 62%
  bars until 3.5% favourable move        n=333
   p10: 1  p25: 2  p50: 3  p75: 7  p90: 11

LOWS
--------------------------------------------------------
  failed pivots, bars until invalidated   n=389
   p10: 2  p25: 3  p50: 6  p75: 14  p90: 23
   within 2 bars: 17%   within 4: 39%   within 8: 57%
  bars until 3.5% favourable move        n=384
   p10: 1  p25: 1  p50: 3  p75: 6  p90: 12


In [8]:
import numpy as np, pandas as pd

H, L, C = df["high"].values, df["low"].values, df["close"].values
N = len(df)
TARGET, SPAN = 3.5, 32

def bars_to_target(bars, kind, on_close):
    """Bars until a 3.5% favourable move, counted the same way each side.
    on_close=True  -> requires a CLOSE beyond the level (same rule as invalidation)
    on_close=False -> a wick touching the level counts (the original rule)
    Search stops at invalidation, matching the original loop."""
    out = []
    for b in bars:
        i = int(b)
        ref = H[i] if kind == "high" else L[i]
        lvl = ref * (1 - TARGET/100) if kind == "high" else ref * (1 + TARGET/100)
        hit = np.nan
        for k in range(1, SPAN + 1):
            j = i + k
            if j >= N:
                break
            if on_close:
                reached = C[j] <= lvl if kind == "high" else C[j] >= lvl
            else:
                reached = L[j] <= lvl if kind == "high" else H[j] >= lvl
            if reached:
                hit = k
                break
            if (C[j] > ref) if kind == "high" else (C[j] < ref):
                break          # invalidated first
        out.append(hit)
    return pd.Series(out, dtype=float)

print("SAME-CLOCK TIMING -- both measured on closes")
print("=" * 70)
print(f"{'':<20}{'n':>6}{'reached':>9}{'p25':>6}{'p50':>6}{'p75':>6}{'p90':>7}")
print("-" * 70)

rows = []
for name, p, kind in [("HIGHS", h15, "high"), ("LOWS", l15, "low")]:
    inval = p.loc[~p["held"], "bars_held"]
    wick  = bars_to_target(p["bar"], kind, on_close=False)
    close = bars_to_target(p["bar"], kind, on_close=True)

    def line(lab, s, denom=None):
        v = s.dropna()
        pct = f"{len(v)/(denom or len(s))*100:>8.0f}%"
        print(f"  {lab:<18}{len(v):>6}{pct}"
              f"{np.percentile(v,25):>6.0f}{np.percentile(v,50):>6.0f}"
              f"{np.percentile(v,75):>6.0f}{np.percentile(v,90):>7.0f}")

    print(f"{name}")
    line("invalidation", inval, len(p))
    line("3.5% on wick", wick, len(p))
    line("3.5% on close", close, len(p))
    rows.append((name, np.median(inval), np.nanmedian(wick), np.nanmedian(close),
                 wick.notna().mean()*100, close.notna().mean()*100))
    print()

print("THE COMPARISON THAT WAS UNFAIR, AND THE FAIR ONE")
print("=" * 70)
print(f"{'':<10}{'invalidate':>12}{'3.5% wick':>12}{'3.5% close':>12}"
      f"{'unfair gap':>12}{'fair gap':>11}")
print("-" * 70)
for name, inv, w, c, _, _ in rows:
    print(f"{name:<10}{inv:>12.0f}{w:>12.0f}{c:>12.0f}"
          f"{inv - w:>+12.0f}{inv - c:>+11.0f}")

print()
print("HOW MANY REACHED THE TARGET AT ALL")
print("-" * 70)
for name, _, _, _, pw, pc in rows:
    print(f"  {name:<8} on a wick {pw:>4.0f}%   on a close {pc:>4.0f}%   "
          f"lost by requiring a close: {pw-pc:>4.0f} points")

SAME-CLOCK TIMING -- both measured on closes
                         n  reached   p25   p50   p75    p90
----------------------------------------------------------------------
HIGHS
  invalidation         431      70%     3     6    12     22
  3.5% on wick         333      54%     2     3     7     11
  3.5% on close        282      46%     2     5     9     15

LOWS
  invalidation         389      63%     3     6    14     23
  3.5% on wick         384      62%     1     3     6     12
  3.5% on close        336      54%     2     4     9     15

THE COMPARISON THAT WAS UNFAIR, AND THE FAIR ONE
            invalidate   3.5% wick  3.5% close  unfair gap   fair gap
----------------------------------------------------------------------
HIGHS                6           3           5          +3         +1
LOWS                 6           3           4          +3         +2

HOW MANY REACHED THE TARGET AT ALL
----------------------------------------------------------------------
  HIGHS

In [9]:
import numpy as np, pandas as pd

H, L, C = df["high"].values, df["low"].values, df["close"].values
N, SPAN = len(df), 32

def reached(bars, kind, target, on_close):
    """True/False per pivot: did a favourable move of `target`% happen
    before invalidation? Search stops at invalidation either way."""
    out = []
    for b in bars:
        i = int(b)
        ref = H[i] if kind == "high" else L[i]
        lvl = ref * (1 - target/100) if kind == "high" else ref * (1 + target/100)
        hit = False
        for k in range(1, SPAN + 1):
            j = i + k
            if j >= N:
                break
            if on_close:
                ok = C[j] <= lvl if kind == "high" else C[j] >= lvl
            else:
                ok = L[j] <= lvl if kind == "high" else H[j] >= lvl
            if ok:
                hit = True
                break
            if (C[j] > ref) if kind == "high" else (C[j] < ref):
                break
        out.append(hit)
    return pd.Series(out, index=bars.index)

TARGETS = (2.0, 2.5, 3.0, 3.5, 4.0)

for name, p, kind, medfail in [("HIGHS", h15, "high", 2.66),
                               ("LOWS",  l15, "low",  2.99)]:
    held = p["held"].values
    print(f"\n{name}   median move on a failure: {medfail}%")
    print("=" * 78)
    print(f"{'target':>7}{'all wick':>10}{'all close':>11}{'wick cost':>11}"
          f"{'held':>9}{'failed':>9}{'spread':>9}")
    print("-" * 78)
    for t in TARGETS:
        w = reached(p["bar"], kind, t, on_close=False)
        c = reached(p["bar"], kind, t, on_close=True)
        wh, wf = w[held].mean()*100, w[~held].mean()*100
        print(f"{t:>7.1f}{w.mean()*100:>9.0f}%{c.mean()*100:>10.0f}%"
              f"{(w.mean()-c.mean())*100:>10.0f}p"
              f"{wh:>8.0f}%{wf:>8.0f}%{wh-wf:>8.0f}p")


HIGHS   median move on a failure: 2.66%
 target  all wick  all close  wick cost     held   failed   spread
------------------------------------------------------------------------------
    2.0       76%        61%        15p     100%      66%      34p
    2.5       67%        55%        12p     100%      53%      47p
    3.0       59%        50%         9p     100%      41%      59p
    3.5       54%        46%         8p     100%      35%      65p
    4.0       50%        41%         9p     100%      28%      72p

LOWS   median move on a failure: 2.99%
 target  all wick  all close  wick cost     held   failed   spread
------------------------------------------------------------------------------
    2.0       81%        71%        10p     100%      70%      30p
    2.5       74%        65%         9p     100%      59%      41p
    3.0       68%        59%        10p     100%      49%      50p
    3.5       62%        54%         8p      99%      39%      60p
    4.0       58%       

In [10]:
import numpy as np, pandas as pd

H, L, C = df["high"].values, df["low"].values, df["close"].values
N, SPAN = len(df), 32

def bars_to(bars, kind, target, on_close):
    out = []
    for b in bars:
        i = int(b)
        ref = H[i] if kind == "high" else L[i]
        lvl = ref * (1 - target/100) if kind == "high" else ref * (1 + target/100)
        hit = np.nan
        for k in range(1, SPAN + 1):
            j = i + k
            if j >= N:
                break
            ok = ((C[j] <= lvl if kind == "high" else C[j] >= lvl) if on_close
                  else (L[j] <= lvl if kind == "high" else H[j] >= lvl))
            if ok:
                hit = k
                break
            if (C[j] > ref) if kind == "high" else (C[j] < ref):
                break
        out.append(hit)
    return pd.Series(out, index=bars.index, dtype=float)

for name, p, kind, tgt in [("HIGHS", h15, "high", 2.66),
                           ("LOWS",  l15, "low",  2.99)]:
    print(f"\n{name} -- bars to reach {tgt}%")
    print("=" * 66)
    print(f"{'':<22}{'n':>6}{'reached':>9}{'p25':>6}{'p50':>6}{'p75':>6}{'p90':>7}")
    print("-" * 66)
    for clock, oc in [("wick", False), ("close", True)]:
        s = bars_to(p["bar"], kind, tgt, oc)
        for lab, mask in [("all", slice(None)),
                          ("held", p["held"].values),
                          ("failed", ~p["held"].values)]:
            v = s[mask].dropna()
            den = len(s[mask])
            print(f"  {clock + ', ' + lab:<20}{len(v):>6}{len(v)/den*100:>8.0f}%"
                  f"{np.percentile(v,25):>6.0f}{np.percentile(v,50):>6.0f}"
                  f"{np.percentile(v,75):>6.0f}{np.percentile(v,90):>7.0f}")
        print()


HIGHS -- bars to reach 2.66%
                           n  reached   p25   p50   p75    p90
------------------------------------------------------------------
  wick, all              400      65%     1     2     4      7
  wick, held             184     100%     1     2     5      8
  wick, failed           216      50%     1     2     3      6

  close, all             320      52%     1     3     6     10
  close, held            184     100%     2     3     7     10
  close, failed          136      32%     1     3     4      8


LOWS -- bars to reach 2.99%
                           n  reached   p25   p50   p75    p90
------------------------------------------------------------------
  wick, all              426      68%     1     2     5     10
  wick, held             232     100%     1     3     6     11
  wick, failed           194      50%     1     2     4      8

  close, all             364      59%     1     3     7     12
  close, held            230      99%     2     

In [11]:
import numpy as np, pandas as pd
from scipy import stats

H, L, C = df["high"].values, df["low"].values, df["close"].values
N, SPAN = len(df), 32
FLAT = [2025, 2026]

def bars_to(p, kind, target, on_close):
    out = []
    for b in p["bar"]:
        i = int(b)
        ref = H[i] if kind == "high" else L[i]
        lvl = ref * (1 - target/100) if kind == "high" else ref * (1 + target/100)
        hit = np.nan
        for k in range(1, SPAN + 1):
            j = i + k
            if j >= N:
                break
            ok = ((C[j] <= lvl if kind == "high" else C[j] >= lvl) if on_close
                  else (L[j] <= lvl if kind == "high" else H[j] >= lvl))
            if ok:
                hit = k; break
            if (C[j] > ref) if kind == "high" else (C[j] < ref):
                break
        out.append(hit)
    return pd.Series(out, index=p.index, dtype=float)

def prop_p(a, b):
    t = [[a.sum(), len(a)-a.sum()], [b.sum(), len(b)-b.sum()]]
    if min(min(r) for r in t) < 1: return float("nan")
    return stats.chi2_contingency(t)[1]

print("BENCHMARK HIT RATES -- full sample vs driftless window (2025-2026)")
print("=" * 78)
print(f"{'':<26}{'n':>6}{'hit%':>7}{'p25':>6}{'p50':>6}{'p75':>6}{'p90':>7}")
print("-" * 78)

store = {}
for name, p, kind, tgt in [("HIGHS", h15, "high", 2.66), ("LOWS", l15, "low", 2.99)]:
    print(f"{name}  target {tgt}%   (wick clock)")
    for label, sub in [("full sample", p), ("flat years", p[p["year"].isin(FLAT)])]:
        s = bars_to(sub, kind, tgt, on_close=False)
        v = s.dropna()
        store[(name, label)] = s.notna()
        print(f"  {label:<24}{len(sub):>6}{len(v)/len(sub)*100:>6.0f}%"
              f"{np.percentile(v,25):>6.0f}{np.percentile(v,50):>6.0f}"
              f"{np.percentile(v,75):>6.0f}{np.percentile(v,90):>7.0f}")
    print()

print("IS THE HIGHS/LOWS DIFFERENCE REAL, OR DRIFT?")
print("=" * 78)
print(f"{'window':<16}{'highs':>9}{'lows':>9}{'gap':>8}{'p':>8}")
print("-" * 78)
for label in ["full sample", "flat years"]:
    a, b = store[("HIGHS", label)], store[("LOWS", label)]
    print(f"{label:<16}{a.mean()*100:>8.0f}%{b.mean()*100:>8.0f}%"
          f"{(b.mean()-a.mean())*100:>+8.1f}{prop_p(a, b):>8.3f}")

print()
print("SAME TARGET BOTH SIDES (2.66%) -- removes the target difference")
print("-" * 78)
print(f"{'window':<16}{'highs':>9}{'lows':>9}{'gap':>8}{'p':>8}")
for label, sel in [("full sample", lambda p: p),
                   ("flat years", lambda p: p[p["year"].isin(FLAT)])]:
    a = bars_to(sel(h15), "high", 2.66, False).notna()
    b = bars_to(sel(l15), "low",  2.66, False).notna()
    print(f"{label:<16}{a.mean()*100:>8.0f}%{b.mean()*100:>8.0f}%"
          f"{(b.mean()-a.mean())*100:>+8.1f}{prop_p(a, b):>8.3f}")

BENCHMARK HIT RATES -- full sample vs driftless window (2025-2026)
                               n   hit%   p25   p50   p75    p90
------------------------------------------------------------------------------
HIGHS  target 2.66%   (wick clock)
  full sample                615    65%     1     2     4      7
  flat years                 331    61%     1     2     4      8

LOWS  target 2.99%   (wick clock)
  full sample                622    68%     1     2     5     10
  flat years                 329    67%     1     3     5     10

IS THE HIGHS/LOWS DIFFERENCE REAL, OR DRIFT?
window              highs     lows     gap       p
------------------------------------------------------------------------------
full sample           65%      68%    +3.4   0.220
flat years            61%      67%    +5.5   0.162

SAME TARGET BOTH SIDES (2.66%) -- removes the target difference
------------------------------------------------------------------------------
window              highs     lows   

In [13]:
import numpy as np, pandas as pd
from scipy import stats
from pivot_auto import find_pivots, add_regime
from pivot_lows import find_lows

H, L, C = df["high"].values, df["low"].values, df["close"].values
N, SPAN, FLAT = len(df), 32, [2025, 2026]
TGT = 2.66                      # same target both sides

def hit(p, kind, target=TGT, on_close=False):
    out = []
    for b in p["bar"]:
        i = int(b)
        ref = H[i] if kind == "high" else L[i]
        lvl = ref * (1 - target/100) if kind == "high" else ref * (1 + target/100)
        got = False
        for k in range(1, SPAN + 1):
            j = i + k
            if j >= N: break
            ok = ((C[j] <= lvl if kind == "high" else C[j] >= lvl) if on_close
                  else (L[j] <= lvl if kind == "high" else H[j] >= lvl))
            if ok: got = True; break
            if (C[j] > ref) if kind == "high" else (C[j] < ref): break
        out.append(got)
    return pd.Series(out, index=p.index)

def pval(a, b):
    t = [[a.sum(), len(a)-a.sum()], [b.sum(), len(b)-b.sum()]]
    if min(min(r) for r in t) < 1: return float("nan")
    return stats.chi2_contingency(t)[1]

def prep(p):
    p = p.copy(); p["year"] = pd.to_datetime(p["date"]).dt.year
    return p[p["year"].isin(FLAT)]

def row(lab, hp, lp):
    a, b = hit(hp, "high"), hit(lp, "low")
    print(f"{lab:>9}{len(hp):>7}{len(lp):>7}{a.mean()*100:>9.0f}%{b.mean()*100:>9.0f}%"
          f"{(b.mean()-a.mean())*100:>+8.1f}{pval(a,b):>8.3f}")

print(f"HIT RATE AT {TGT}% BOTH SIDES -- driftless window, full battery")
print("=" * 74)

print("\nATR THRESHOLD")
print(f"{'mult':>9}{'n_hi':>7}{'n_lo':>7}{'highs':>10}{'lows':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
for m in (1.5, 2.0, 2.5, 3.0, 4.0):
    row(f"{m:.1f}", prep(find_pivots(df, atr_mult=m)), prep(find_lows(df, atr_mult=m)))

print("\nPIVOT SEPARATION  (atr_mult=1.5)")
print(f"{'min_sep':>9}{'n_hi':>7}{'n_lo':>7}{'highs':>10}{'lows':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
for s in (2, 3, 4, 6, 8):
    row(f"{s}", prep(find_pivots(df, atr_mult=1.5, min_sep=s)),
        prep(find_lows(df, atr_mult=1.5, min_sep=s)))

hb = find_pivots(df, atr_mult=1.5); lb = find_lows(df, atr_mult=1.5)
hb["year"] = pd.to_datetime(hb["date"]).dt.year
lb["year"] = pd.to_datetime(lb["date"]).dt.year

print("\nEACH YEAR SEPARATELY  (all four, not just flat)")
print(f"{'year':>9}{'n_hi':>7}{'n_lo':>7}{'highs':>10}{'lows':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
for y in sorted(set(hb["year"]) & set(lb["year"])):
    row(f"{y}", hb[hb["year"]==y], lb[lb["year"]==y])

print("\nBY REGIME  (driftless window)")
print(f"{'regime':>9}{'n_hi':>7}{'n_lo':>7}{'highs':>10}{'lows':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
hr, lr = prep(add_regime(hb, df)), prep(add_regime(lb, df))
for r in ["up", "consolidation", "down"]:
    a, b = hr[hr["regime"]==r], lr[lr["regime"]==r]
    if len(a) >= 20 and len(b) >= 20: row(r, a, b)

print("\nON A CLOSE INSTEAD OF A WICK  (driftless window)")
print("-" * 74)
hp, lp = prep(hb), prep(lb)
a, b = hit(hp, "high", on_close=True), hit(lp, "low", on_close=True)
print(f"{'close':>9}{len(hp):>7}{len(lp):>7}{a.mean()*100:>9.0f}%{b.mean()*100:>9.0f}%"
      f"{(b.mean()-a.mean())*100:>+8.1f}{pval(a,b):>8.3f}")

print("\nOTHER TARGETS  (driftless window, wick)")
print(f"{'target':>9}{'n_hi':>7}{'n_lo':>7}{'highs':>10}{'lows':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
for t in (2.0, 2.5, 3.0, 3.5, 4.0):
    a, b = hit(hp, "high", target=t), hit(lp, "low", target=t)
    print(f"{t:>9.1f}{len(hp):>7}{len(lp):>7}{a.mean()*100:>9.0f}%{b.mean()*100:>9.0f}%"
          f"{(b.mean()-a.mean())*100:>+8.1f}{pval(a,b):>8.3f}")

HIT RATE AT 2.66% BOTH SIDES -- driftless window, full battery

ATR THRESHOLD
     mult   n_hi   n_lo     highs      lows     gap       p
--------------------------------------------------------------------------
      1.5    331    329       61%       71%   +10.4   0.006
      2.0    228    212       61%       73%   +11.2   0.016
      2.5    165    154       63%       73%    +9.7   0.083
      3.0    125    115       65%       73%    +8.2   0.216
      4.0     69     66       62%       79%   +16.5   0.057

PIVOT SEPARATION  (atr_mult=1.5)
  min_sep   n_hi   n_lo     highs      lows     gap       p
--------------------------------------------------------------------------
        2    331    329       61%       71%   +10.4   0.006
        3    309    304       61%       70%    +9.2   0.020
        4    287    282       62%       70%    +7.5   0.073
        6    233    233       61%       69%    +8.2   0.081
        8    208    201       59%       68%    +9.0   0.073

EACH YEAR SEPARAT